# 1. Cassava, the dataset, and this workshop

## Why cassava leaf disease matters

Cassava is an important food crop in many tropical and subtropical regions,
including many African countries. Cassava plants can be affected by leaf
diseases that reduce crop yield. Identifying disease symptoms from leaf images
can support earlier diagnosis and agricultural decision-making.

This workshop uses **Cassava training metadata**. Each record links an image
reference to a class label describing a cassava leaf disease category or a
healthy leaf.

> **Educational purpose:** this is a federated-learning workflow tutorial. It
> is not an agricultural diagnosis tool and does not validate a model for use
> by farmers, agronomists, or decision makers.

## The workshop metadata

The current workshop source metadata contains **5,656 labelled records** and
two fields:

| Field | Meaning |
|---|---|
| `image` | Reference to a cassava leaf image |
| `label` | Numeric label for the disease or healthy-leaf class |

The labels are:

| Label | Class | Records in source metadata | Share |
|---:|---|---:|---:|
| 0 | Cassava Bacterial Blight (CBB) | 466 | 8.24% |
| 1 | Cassava Brown Streak Disease (CBSD) | 1,443 | 25.51% |
| 2 | Cassava Green Mite (CGM) | 773 | 13.67% |
| 3 | Cassava Mosaic Disease (CMD) | 2,658 | 46.99% |
| 4 | Healthy | 316 | 5.59% |

The classes are not equally represented. In particular, Cassava Mosaic Disease
is the largest class in this workshop metadata. This matters when interpreting
model metrics later.


## What problem are we studying?

In a real cassava image-classification task, a model receives a **leaf image**
as input and predicts one of the five disease/health classes listed above. This
is a **multi-class classification** problem.

### Important scope of this introductory exercise

The current tutorial does **not** open or analyse image pixels. It uses the
cassava metadata and labels to create a deterministic synthetic numeric
representation for each local record. This keeps the first exercise small,
reproducible, and focused on the federated-learning workflow.

Therefore, the accuracy and loss shown later measure performance on this
controlled demonstration task. They do **not** show that the system can
recognise cassava leaf diseases from real images.


## Why use federated learning with this open dataset?

The cassava metadata used in this workshop is openly available. Federated
learning is therefore **not essential** for this particular dataset: the data
could be collected in one place and used for conventional centralised machine
learning.

We partition the data to demonstrate the workflow used when data belongs to
separate organisations and raw records cannot, or should not, be copied to one
central location.

| Workshop demonstration | Real federated-learning setting |
|---|---|
| Open data is divided into partitions for teaching | Data is already held by separate organisations |
| Partitions are deliberately stratified and similar | Local datasets may differ strongly in size, quality, and class distribution |
| Goal: understand the technical workflow | Goal: collaboratively train without centralising raw data |
| Two participant groups | Farms, cooperatives, research stations, companies, hospitals, or public authorities |

Federated learning is **not** a guarantee of privacy or security. Real-world
deployments may also require governance agreements, access controls, secure
aggregation, privacy-preserving methods, and careful model evaluation.


## Optional learning materials — choose what helps you

The workshop is fully self-contained: you do **not** need to watch or read
these resources to complete the notebooks. They are available if you would
like another explanation, an example, or more background.

### Short conceptual videos

1. **[What is Federated Learning? | Federated Learning in Machine Learning](https://www.youtube.com/watch?v=Ero4yY_ttJE)**  
   Intellipaat — approximately 13 minutes. A short introduction to federated
   learning and the difference from centralised machine learning.

2. **[Federated Learning Explained: Frameworks, Applications, Challenges & Real-World Examples](https://www.youtube.com/watch?v=EcgcNn_TBw8)**  
   The Tech Spark — approximately 13 minutes. A visual overview of the basic
   workflow, applications, and practical challenges.

### A longer structured course

3. **[Intro to Federated Learning](https://learn.deeplearning.ai/courses/intro-to-federated-learning/lesson/uhuz2/introduction)**  
   DeepLearning.AI — approximately 1.5 hours in total, organised as shorter
   lessons with exercises.

> **Use the notebooks as the source of truth for this workshop.** In
> particular, this exercise uses synthetic features rather than leaf-image
> pixels, and federated learning does not automatically guarantee privacy,
> security, or suitable data governance.


## Explore your group’s local data partition

## Goal

In this notebook, you will examine the data available to **your group**
before federated learning begins.

Your group has one local partition. Other groups’ partitions are not
mounted in this workspace and are not inspected here.

**Run the cells from top to bottom.** The workshop software is already
prepared before this notebook opens; you do not need to install anything.


## What you will learn

By the end of this notebook, you should be able to explain:

1. what the cassava metadata records represent;
2. which cassava disease/health classes occur in the workshop data;
3. which group you represent and how many examples are in its local partition;
4. how labels are distributed in that partition;
5. why this tutorial uses a controlled synthetic representation rather than
   opening image files; and
6. why federated learning is demonstrated even though this workshop data is open.


In [ ]:
import os
import sys
from pathlib import Path

import flwr
import pandas as pd

group_id = os.environ["GROUP_ID"]
workshop_root = Path(os.environ["DIGITAFRICA_WORKSHOP_ROOT"])
data_path = Path(os.environ["CLIENT_DATA_PATH"])
app_root = workshop_root / "app"

if not group_id.startswith("group_"):
    raise RuntimeError(
        f"This notebook requires a workshop group login, got {group_id!r}."
    )
if not data_path.is_file():
    raise FileNotFoundError(f"Local partition is unavailable: {data_path}")
if str(app_root) not in sys.path:
    sys.path.insert(0, str(app_root))

print("Workshop environment is ready.")
print(f"Your group:          {group_id}")
print(f"Your local partition: {data_path.name}")
print(f"Flower version:       {flwr.__version__}")
print("Only your group's prepared CSV metadata is available in this notebook.")


## Step 1 — Read local metadata

Each row represents one example. The `image` value is an identifier;
this tutorial does **not** open the source image file. The `label` value
is the class associated with that example.


In [ ]:
LABEL_NAMES = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mite (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy",
}

partition = pd.read_csv(data_path)

required_columns = {"image", "label"}
missing_columns = required_columns.difference(partition.columns)
if missing_columns:
    raise ValueError(
        f"Partition is missing required columns: {sorted(missing_columns)}"
    )

partition["label"] = pd.to_numeric(
    partition["label"], errors="raise"
).astype(int)

unknown_labels = sorted(set(partition["label"]).difference(LABEL_NAMES))
if unknown_labels:
    raise ValueError(f"Unknown labels in local partition: {unknown_labels}")

summary = (
    partition["label"]
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .reset_index(name="examples")
)
summary["class"] = summary["label"].map(LABEL_NAMES)
summary["percentage"] = (100 * summary["examples"] / len(partition)).round(2)
summary = summary[["label", "class", "examples", "percentage"]]

print(f"Examples in {group_id}'s local partition: {len(partition)}")
print("\nLocal class distribution:")
display(summary)

preview = partition[["image", "label"]].head().copy()
preview["image"] = preview["image"].map(lambda value: Path(str(value)).name)
preview["class"] = preview["label"].map(LABEL_NAMES)
preview = preview.rename(columns={"image": "image filename"})

print("\nFirst five metadata records:")
print("For readability, only the image filename is shown; source paths are hidden.")
display(preview)


## Step 2 — Prepare a controlled federated-learning workflow demonstration

A real cassava image model would need access to image pixels and an approved
modelling and evaluation protocol. This workshop instead demonstrates the
*federated-learning workflow*.

It derives deterministic synthetic numeric features from local metadata. This
lets you observe local training, model-update sharing, and server aggregation
without claiming that the result is a validated cassava disease classifier.


In [ ]:
from client.client import load_partition

features, labels = load_partition(
    data_path,
    num_classes=5,
    feature_dim=16,
    seed=42,
)

print("Synthetic representation prepared successfully.")
print(f"Feature matrix: {features.shape[0]} local examples × {features.shape[1]} features")
print(f"Label vector:   {labels.shape[0]} local labels")
print(f"Feature type:   {features.dtype}")
print(
    "\nThe feature matrix remains inside this group workspace. "
    "Only model updates are exchanged during federation."
)


## Ready for federation

You have now checked the local partition used by your group.

**Do not start a client yet.** Wait for the organiser to confirm that all
groups are ready. Then open `02_Run_Federated_Client.ipynb`.

**Check your understanding:** Why is it useful that each group can inspect
its own label distribution before training starts?
